# 🛠️ Урок 22 — Модель и интерфейс (материалы преподавателя)

Улучшаем модель → честно оцениваем → Gradio → README и слайды.

> Пример на Palmer Penguins. Первичная модель `model` восстанавливается в Шаге 0 (ноутбук самодостаточный).

## Шаг 0 · Восстановим первичную модель с урока 21 (пример)

In [ ]:
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
df = sns.load_dataset('penguins').drop(columns=['sex'])   # 👈 пример; sex всегда убираем
df = df.dropna(subset=['species'])
num = ['bill_length_mm','bill_depth_mm','flipper_length_mm','body_mass_g']  # 👈 свои числовые
cat = ['island']                                           # 👈 свои категориальные
X = df[num+cat]; y = df['species']                         # 👈 свой target
X_tr,X_te,y_tr,y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
prep = ColumnTransformer([
    ('num', Pipeline([('i',SimpleImputer(strategy='median')),('s',StandardScaler())]), num),
    ('cat', Pipeline([('i',SimpleImputer(strategy='most_frequent')),('o',OneHotEncoder(handle_unknown='ignore'))]), cat)])
model = Pipeline([('prep',prep),('rf',RandomForestClassifier(n_estimators=100,random_state=42))])
model.fit(X_tr, y_tr)
print(f'Первичная модель: {accuracy_score(y_te, model.predict(X_te)):.0%}')

## Шаг 1 · Улучшение + честная оценка
Подбор параметров (GridSearchCV), метрика под задачу, кросс-валидация.

In [ ]:
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.metrics import classification_report
grid = GridSearchCV(model, {'rf__n_estimators':[100,300], 'rf__max_depth':[None,5,10]}, cv=5)
grid.fit(X_tr, y_tr)
best = grid.best_estimator_
print('Лучшие параметры:', grid.best_params_)
print(classification_report(y_te, best.predict(X_te)))
print(f'Кросс-валидация: {cross_val_score(best, X, y, cv=5).mean():.1%}')

## Шаг 2 (в Colab) · Веб-интерфейс через Gradio
⚠️ `Pipeline` в модели — это **sklearn Pipeline**, не HF `pipeline`. Входы интерфейса должны совпадать с признаками модели.

In [ ]:
!pip install gradio -q
import gradio as gr, pandas as pd
def predict(bill_len, bill_dep, flipper, mass, island):
    row = pd.DataFrame([{'bill_length_mm':bill_len,'bill_depth_mm':bill_dep,
                         'flipper_length_mm':flipper,'body_mass_g':mass,'island':island}])
    return str(best.predict(row)[0])
gr.Interface(fn=predict,
    inputs=[gr.Number(label='Длина клюва, мм'), gr.Number(label='Глубина клюва, мм'),
            gr.Number(label='Длина ласта, мм'), gr.Number(label='Масса, г'),
            gr.Radio(['Biscoe','Dream','Torgersen'], label='Остров')],
    outputs=gr.Label(label='Вид пингвина'), title='Мой проект').launch(share=True)

## Шаг 3 · README и структура презентации
**README.md:**
```markdown
# Название проекта
## Задача
Что предсказываем, метрика.
## Данные
Источник, размер, признаки.
## Модель
Какая, в Pipeline, сравнение с baseline.
## Результаты
Метрика на test/CV, где ошибается.
## Как запустить
Colab / ссылка на приложение.
```

**Слайды (3–5):** задача → данные → модель и метрика → демо → выводы.

---
**Итог урока 22.** Модель улучшена и честно оценена, есть интерфейс, готова структура рассказа. Дальше (урок 23) — репетиция и финальный деплой.